### Ingestión la carpeta production_contry

In [0]:
%run '../includes/configuration'

In [0]:
%run '../includes/common_functions'

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date","2024-12-23")
v_file_date = dbutils.widgets.get("p_file_date")

##### Paso 1. Leer los archivos

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
productions_countries_schema = StructType(fields=[
    StructField("movieId", IntegerType(), True),
    StructField("countryId", IntegerType(), True)
])

In [0]:
productions_countries_df = spark.read \
                .schema(productions_countries_schema) \
                .option("multiline",True) \
                .json(f"{bronze_folder_path}/{v_file_date}/production_country/*.json")

##### Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import col, concat, current_timestamp, lit, trim

In [0]:
productions_countries_final_df = productions_countries_df \
                        .withColumnRenamed("movieId","movie_id") \
                        .withColumnRenamed("countryId","country_id") \
                        .withColumn("ingestion_date", current_timestamp()) \
                        .withColumn("environment", lit(v_environment))\
                        .withColumn("file_date", lit(v_file_date))

#####Paso 3 - Escribir la salida en formato parquet

In [0]:
#overwrite_partition("movie_silver","productions_countries","file_date",v_file_date)

In [0]:
merge_condition = "tgt.movie_id = src.movie_id AND tgt.country_id = src.country_id AND tgt.file_date = src.file_date"
merge_delta_lake(productions_countries_final_df,"movie_silver","productions_countries",merge_condition,"file_date")

In [0]:
%sql 
SELECT file_date, COUNT(1) FROM movie_silver.productions_countries
GROUP BY file_date 

file_date,count(1)
2024-12-16,3750
2024-12-23,1250
2024-12-30,1436
